# Real-Time Fraud Detection Engine
**IEEE-CIS Dataset · LightGBM + XGBoost Ensemble**

A locally-trained, production-inspired fraud detection system featuring:
- Time-safe feature engineering with user-level aggregations
- Gradient boosted ensemble with optimal weight search
- Isolation Forest trained (excluded from final ensemble due to low PR-AUC)
- Adaptive threshold engine (bias-corrected, dual-signal design)
- Business rules layer with structured flagging reasons
- Async streaming engine at 1000+ TPS
- SHAP explainability on every flagged transaction

> **Environment:** 100% local · No cloud · IEEE-CIS (590K transactions)

## 1. Imports

In [2]:
import pandas as pd
import numpy as np
import joblib, os, json, time, warnings
from collections import deque
from dataclasses import dataclass, field

from lightgbm import LGBMClassifier, early_stopping
import xgboost as xgb
from sklearn.ensemble import IsolationForest
from sklearn.metrics import (
    average_precision_score, f1_score,
    precision_score, recall_score
)

warnings.filterwarnings('ignore')
print("Libraries loaded.")

Libraries loaded.


## 2. Load Data
IEEE-CIS Fraud Detection dataset from Kaggle.  
Two tables: `train_transaction` (payment features) and `train_identity` (device/browser features).

In [3]:
train_transaction = pd.read_csv("../data/train_transaction.csv")
train_identity    = pd.read_csv("../data/train_identity.csv")

print("train_transaction:", train_transaction.shape)
print("train_identity:   ", train_identity.shape)

train_transaction: (590540, 394)
train_identity:    (144233, 41)


## 3. Merge
Left-join on `TransactionID`. Identity features are sparse (~40% match rate), which is fine — missingness itself is a signal.

In [4]:
train = train_transaction.merge(train_identity, on='TransactionID', how='left')
print("Merged:", train.shape)
print(f"Fraud rate: {train['isFraud'].mean():.4f}  ({train['isFraud'].sum():,} fraud / {len(train):,} total)")

Merged: (590540, 434)
Fraud rate: 0.0350  (20,663 fraud / 590,540 total)


## 4. Preprocessing
Encode boolean M-columns and cast categoricals **on the full dataset** before the train/val split.  
This ensures both splits share identical category levels - required by LightGBM's native categorical support.

In [5]:
# M1-M3, M5, M7-M9: T/F → 1/0
for col in ['M1', 'M2', 'M3', 'M5', 'M7', 'M8', 'M9']:
    train[col] = train[col].map({'T': 1, 'F': 0})

# Cast categoricals on the full frame
for col in ['ProductCD', 'card4', 'card6', 'DeviceType',
            'id_30', 'id_31', 'P_emaildomain', 'R_emaildomain', 'M4', 'M6']:
    train[col] = train[col].astype('category')

# DeviceBrand: first token of DeviceInfo
train['DeviceBrand'] = (
    train['DeviceInfo']
    .fillna('missing')
    .str.split()
    .str[0]
    .astype('category')
)

print("Preprocessing done.")
print("DeviceBrand unique values:", train['DeviceBrand'].nunique())

Preprocessing done.
DeviceBrand unique values: 1183


## 5. Time-Based Train / Val Split
**Critical design choice:** split on `TransactionDT` (not random).  
Random splits leak future transactions into training - a common mistake that inflates metrics and doesn't reflect production behaviour.

In [6]:
threshold = train['TransactionDT'].quantile(0.80)

X_train = train[train['TransactionDT'] < threshold].copy()
X_val   = train[train['TransactionDT'] >= threshold].copy()

print(f"X_train : {X_train.shape}  |  fraud rate: {X_train['isFraud'].mean():.4f}")
print(f"X_val   : {X_val.shape}  |  fraud rate: {X_val['isFraud'].mean():.4f}")

X_train : (472432, 435)  |  fraud rate: 0.0351
X_val   : (118108, 435)  |  fraud rate: 0.0344


## 6. Feature Engineering
All aggregation maps are computed **from X_train only**.  
X_val receives mapped values with `fillna` fallback - no leakage from future data.

Features engineered:
- `uid` (card1 + addr1): user-level transaction count and amount statistics
- `uid2` (card1+card2+card3+card5): finer card identity aggregations
- `uid_d1`: card + normalised account-open day for card-age signals
- `D1n`: days since card was first seen (derived from TransactionDT and D1)
- Per-uid mean of C-columns (count features used in fraud detection)

In [7]:
# ── uid = card1 + addr1 ────────────────────────────────────────────────────────
X_train['uid'] = X_train['card1'].astype(str) + '_' + X_train['addr1'].astype(str)
X_val['uid']   = X_val['card1'].astype(str)   + '_' + X_val['addr1'].astype(str)

uid_count_map    = X_train.groupby('uid').size()
uid_amt_mean_map = X_train.groupby('uid')['TransactionAmt'].mean()
uid_amt_std_map  = X_train.groupby('uid')['TransactionAmt'].std()

X_train['uid_count']    = X_train['uid'].map(uid_count_map)
X_val['uid_count']      = X_val['uid'].map(uid_count_map).fillna(0)

X_train['uid_amt_mean'] = X_train['uid'].map(uid_amt_mean_map)
X_val['uid_amt_mean']   = X_val['uid'].map(uid_amt_mean_map).fillna(X_train['TransactionAmt'].mean())

X_train['uid_amt_std']  = X_train['uid'].map(uid_amt_std_map).fillna(0)
X_val['uid_amt_std']    = X_val['uid'].map(uid_amt_std_map).fillna(0)

for col in ['C1', 'C2', 'C13', 'C14']:
    m = X_train.groupby('uid')[col].mean()
    X_train[f'uid_{col}_mean'] = X_train['uid'].map(m)
    X_val[f'uid_{col}_mean']   = X_val['uid'].map(m).fillna(X_train[col].mean())

# ── D1n: normalised days since card first seen ──────────────────────────────
for df in [X_train, X_val]:
    df['D1n'] = np.floor(df['TransactionDT'] / (24 * 60 * 60)) - df['D1']

# ── uid2 = card1+card2+card3+card5 ─────────────────────────────────────────
for df in [X_train, X_val]:
    df['uid2'] = (df['card1'].astype(str) + '_' + df['card2'].astype(str) + '_' +
                  df['card3'].astype(str) + '_' + df['card5'].astype(str))

uid2_amt_mean_map = X_train.groupby('uid2')['TransactionAmt'].mean()
uid2_amt_std_map  = X_train.groupby('uid2')['TransactionAmt'].std()

X_train['uid2_amt_mean'] = X_train['uid2'].map(uid2_amt_mean_map)
X_val['uid2_amt_mean']   = X_val['uid2'].map(uid2_amt_mean_map).fillna(X_train['TransactionAmt'].mean())

X_train['uid2_amt_std']  = X_train['uid2'].map(uid2_amt_std_map).fillna(0)
X_val['uid2_amt_std']    = X_val['uid2'].map(uid2_amt_std_map).fillna(0)

# ── uid_d1: card1 + normalised D1 day ──────────────────────────────────────
for df in [X_train, X_val]:
    df['uid_d1'] = (df['card1'].astype(str) + '_' +
                    np.floor(df['TransactionDT'] / (24 * 60 * 60) - df['D1']).astype(str))

uid_d1_amt_mean_map = X_train.groupby('uid_d1')['TransactionAmt'].mean()
X_train['uid_d1_amt_mean'] = X_train['uid_d1'].map(uid_d1_amt_mean_map)
X_val['uid_d1_amt_mean']   = X_val['uid_d1'].map(uid_d1_amt_mean_map).fillna(X_train['TransactionAmt'].mean())

print("Feature engineering done.")

Feature engineering done.


## 7. Feature List
Defined exactly once. This list is the single source of truth for model training, inference pipeline, and the FastAPI backend.

In [8]:
features = [
    'TransactionAmt',
    'ProductCD', 'card4', 'card6',
    'DeviceType', 'id_30', 'id_31',
    'P_emaildomain', 'R_emaildomain',
    'addr1', 'addr2',
    'M4', 'M6',
    'V188', 'V189', 'V200', 'V201', 'V242', 'V244', 'V246', 'V257', 'V258',
    'V147', 'V148', 'V149', 'V154', 'V155', 'V156', 'V157', 'V158',
    'D1', 'D2', 'D3', 'D4', 'D5', 'D10', 'D11', 'D15',
    'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8',
    'C9', 'C10', 'C11', 'C12', 'C13', 'C14',
    'M1', 'M2', 'M3', 'M5', 'M7', 'M8', 'M9',
    'V39', 'V40', 'V42', 'V43', 'V44', 'V45',
    'V170', 'V171',
    'V46', 'V47', 'V48', 'V49', 'V50', 'V51', 'V52',
    'id_01', 'id_02', 'id_03', 'id_04', 'id_05', 'id_06',
    'id_09', 'id_10', 'id_11', 'id_13', 'id_17', 'id_20',
    'DeviceBrand',
    'uid_count', 'uid_amt_mean', 'uid_amt_std',
    'uid_C1_mean', 'uid_C2_mean', 'uid_C13_mean', 'uid_C14_mean',
    'D1n',
    'uid2_amt_mean', 'uid2_amt_std',
    'uid_d1_amt_mean',
]

assert len(features) == len(set(features)), "Duplicate features detected!"
missing = [f for f in features if f not in X_train.columns]
assert not missing, f"Missing from X_train: {missing}"
print(f"Features: {len(features)} — unique ✓ — all present in X_train ✓")

Features: 98 — unique ✓ — all present in X_train ✓


## 8. Categorical Alignment
Force X_val category levels to exactly match X_train.  
Without this, LightGBM raises a `categorical_feature do not match` error at inference time.

In [9]:
for col in features:
    if str(X_train[col].dtype) == 'category':
        X_val[col] = pd.Categorical(X_val[col], categories=X_train[col].cat.categories)

cat_in_features = [c for c in features if str(X_train[c].dtype) == 'category']
print(f"Categorical features ({len(cat_in_features)}): {cat_in_features}")

Categorical features (11): ['ProductCD', 'card4', 'card6', 'DeviceType', 'id_30', 'id_31', 'P_emaildomain', 'R_emaildomain', 'M4', 'M6', 'DeviceBrand']


## 9. Train LightGBM
Gradient boosted trees with leaf-wise growth.  
Early stopping on `average_precision` (PR-AUC) - the correct metric for imbalanced fraud data where precision-recall trade-off matters more than accuracy.

In [10]:
lgb_model = LGBMClassifier(
    n_estimators=3000,
    learning_rate=0.01,
    num_leaves=256,
    subsample=0.7,
    colsample_bytree=0.7,
    random_state=42,
    n_jobs=-1
)

lgb_model.fit(
    X_train[features],
    X_train['isFraud'],
    eval_set=[(X_val[features], X_val['isFraud'])],
    eval_metric='average_precision',
    callbacks=[early_stopping(100)]
)

lgb_preds = lgb_model.predict_proba(X_val[features])[:, 1]
lgb_prauc = average_precision_score(X_val['isFraud'], lgb_preds)

print(f"LightGBM best iteration : {lgb_model.best_iteration_}")
print(f"LightGBM PR-AUC         : {lgb_prauc:.6f}")

[LightGBM] [Info] Number of positive: 16599, number of negative: 455833
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.027075 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 10586
[LightGBM] [Info] Number of data points in the train set: 472432, number of used features: 98
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.035135 -> initscore=-3.312784
[LightGBM] [Info] Start training from score -3.312784
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1122]	valid_0's average_precision: 0.622679	valid_0's binary_logloss: 0.0785981
LightGBM best iteration : 1122
LightGBM PR-AUC         : 0.622679


## 10. Train XGBoost
Depth-wise growth (different inductive bias from LightGBM).  
Both models trained on the same features - their different error patterns is what makes the ensemble effective.

In [11]:
xgb_model = xgb.XGBClassifier(
    n_estimators=3000,
    learning_rate=0.02,
    max_depth=12,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.4,
    eval_metric='aucpr',
    random_state=42,
    tree_method='hist',
    enable_categorical=True
)

xgb_model.fit(X_train[features], X_train['isFraud'])

xgb_preds = xgb_model.predict_proba(X_val[features])[:, 1]
xgb_prauc = average_precision_score(X_val['isFraud'], xgb_preds)

print(f"XGBoost PR-AUC : {xgb_prauc:.6f}")

XGBoost PR-AUC : 0.620104


## 11. Isolation Forest - Trained but Not Used in Final Ensemble
Isolation Forest is trained here for completeness and saved to disk.

However, its PR-AUC on the validation set is ~0.05-0.10, which is expected for an unsupervised model on a labelled dataset. Including it in the ensemble **reduces overall PR-AUC** rather than improving it.

**Decision:** The final ensemble uses LightGBM + XGBoost only. ISO is excluded from inference.

In [12]:
# IF cannot handle category dtype — use numeric features only
numeric_features = [
    f for f in features
    if str(X_train[f].dtype) not in ('category', 'object')
]

train_medians = X_train[numeric_features].median()

X_train_if = X_train[numeric_features].fillna(train_medians)
X_val_if   = X_val[numeric_features].fillna(train_medians)

iso_model = IsolationForest(
    n_estimators=200,
    contamination=0.035,
    random_state=42,
    n_jobs=-1
)
iso_model.fit(X_train_if)

raw_scores = iso_model.decision_function(X_val_if)
iso_scores = 1 - (raw_scores - raw_scores.min()) / (raw_scores.max() - raw_scores.min())

iso_prauc = average_precision_score(X_val['isFraud'], iso_scores)
print(f"Isolation Forest PR-AUC : {iso_prauc:.6f}")
print("→ PR-AUC is too low to contribute positively to the ensemble.")
print("→ ISO will be saved but excluded from inference and the final ensemble.")

Isolation Forest PR-AUC : 0.241463
→ PR-AUC is too low to contribute positively to the ensemble.
→ ISO will be saved but excluded from inference and the final ensemble.


## 12. Ensemble & Weight Search
Grid-search over LightGBM / XGBoost blend weights to maximise PR-AUC on the validation set.

Isolation Forest is excluded - its inclusion reduces ensemble PR-AUC due to low signal quality.  
Weights sum to 1.0 across LGB + XGB only.

In [13]:
best_score = -1
best_w_lgb = 0.5

for w_lgb in np.arange(0.20, 0.81, 0.01):
    w_xgb  = 1.0 - w_lgb
    preds  = w_lgb * lgb_preds + w_xgb * xgb_preds
    score  = average_precision_score(X_val['isFraud'], preds)
    if score > best_score:
        best_score = score
        best_w_lgb = w_lgb

W_LGB = best_w_lgb
W_XGB = 1.0 - W_LGB

ensemble_preds = W_LGB * lgb_preds + W_XGB * xgb_preds
ensemble_prauc = average_precision_score(X_val['isFraud'], ensemble_preds)

print(f"LightGBM  PR-AUC : {lgb_prauc:.6f}   weight → {W_LGB:.2f}")
print(f"XGBoost   PR-AUC : {xgb_prauc:.6f}   weight → {W_XGB:.2f}")
print(f"Ensemble  PR-AUC : {ensemble_prauc:.6f}  ← final (LGB + XGB only)")

LightGBM  PR-AUC : 0.622679   weight → 0.45
XGBoost   PR-AUC : 0.620104   weight → 0.55
Ensemble  PR-AUC : 0.633571  ← final (LGB + XGB only)


## 13. Save Models & Artefacts
Saves all models, the feature list, numeric feature list, and training medians.  
The FastAPI backend loads these at startup.

In [14]:
joblib.dump(lgb_model,        'lgb_F.pkl')
joblib.dump(xgb_model,        'xgb_F.pkl')
joblib.dump(iso_model,        'iso_F.pkl')       # saved but not used in inference
joblib.dump(features,         'features_F.pkl')
joblib.dump(numeric_features, 'numeric_features_F.pkl')

with open('train_medians.json', 'w') as f:
    json.dump(train_medians.to_dict(), f)

cat_cols = {
    col: list(X_train[col].cat.categories)
    for col in features
    if str(X_train[col].dtype) == 'category'
}
with open('cat_cols.json', 'w') as f:
    json.dump(cat_cols, f)

for fname in ['lgb_F.pkl', 'xgb_F.pkl', 'iso_F.pkl',
              'features_F.pkl', 'numeric_features_F.pkl',
              'train_medians.json', 'cat_cols.json']:
    size_mb = os.path.getsize(fname) / 1e6
    print(f"  {fname}: {size_mb:.1f} MB")

print("\nNote: iso_F.pkl is saved for reproducibility but is NOT loaded in the inference pipeline.")

  lgb_F.pkl: 32.4 MB
  xgb_F.pkl: 53.8 MB
  iso_F.pkl: 1.4 MB
  features_F.pkl: 0.0 MB
  numeric_features_F.pkl: 0.0 MB
  train_medians.json: 0.0 MB
  cat_cols.json: 0.0 MB

Note: iso_F.pkl is saved for reproducibility but is NOT loaded in the inference pipeline.


## 14. Adaptive Threshold Engine
**Design rationale:**  
The naive approach - adapting threshold only from analyst-reviewed alerts - introduces selection bias. It measures *alert precision* (fraud confirmed / alerts reviewed), not *stream fraud prevalence* (fraud / all transactions). A spike in analyst confirmations may simply reflect that analysts are reviewing high-score alerts, not that the underlying fraud rate has changed.

**Corrected design uses two independent, unbiased signals:**

| Signal | What it measures | Data source |
|--------|-----------------|-------------|
| Flag rate | Fraction of all transactions flagged by the model | Every transaction via `record_transaction()` |
| Alert precision | Fraction of flagged alerts confirmed as fraud by analysts | Analyst reviews via `record_review()` |

**Threshold logic:**
- Flag rate > 2× baseline → model is over-triggering → raise threshold
- Alert precision < 30% → too many false positives → raise threshold  
- Alert precision > 85% → model is accurate and has headroom → lower threshold slightly
- **Low flag rate alone does NOT lower the threshold** - fraud may have genuinely decreased; we have no evidence of missed fraud

In [15]:
class AdaptiveThresholdEngine:
    """
    Adapts the fraud score threshold using two unbiased, observable signals.

    Signal 1 — Flag Rate (stream-level):
        flag_rate = flagged / total_txns over a rolling window.
        A high flag rate means the model is over-triggering → raise threshold.
        A low flag rate does NOT lower the threshold: fraud may have genuinely
        decreased and we cannot observe missed fraud from this signal alone.

    Signal 2 — Alert Precision (analyst-level):
        precision = confirmed_fraud / total_reviewed.
        Low precision (many false positives) → raise threshold.
        Consistently high precision → lower threshold slightly (we have headroom).
    """

    BASE_THRESHOLD = 0.50
    MIN_THRESHOLD  = 0.30
    MAX_THRESHOLD  = 0.65

    def __init__(
        self,
        baseline_flag_rate: float = 0.035,
        stream_window: int = 1000,
        review_window: int = 200,
    ):
        self.baseline_flag_rate = baseline_flag_rate
        self._stream  = deque(maxlen=stream_window)   # 1 = flagged, 0 = not
        self._reviews = deque(maxlen=review_window)   # 1 = fraud, 0 = FP

    def record_transaction(self, flagged: bool) -> None:
        """Call for every transaction that passes through the scoring pipeline."""
        self._stream.append(1 if flagged else 0)

    def record_review(self, is_fraud: bool) -> None:
        """Call when an analyst submits a review decision."""
        self._reviews.append(1 if is_fraud else 0)

    @property
    def _flag_rate(self):
        if len(self._stream) < 100:
            return None
        return sum(self._stream) / len(self._stream)

    @property
    def _precision(self):
        if len(self._reviews) < 20:
            return None
        return sum(self._reviews) / len(self._reviews)

    @property
    def threshold(self) -> float:
        thr = self.BASE_THRESHOLD

        fr = self._flag_rate
        if fr is not None:
            ratio = fr / self.baseline_flag_rate
            if ratio > 2.0:
                thr += 0.10
            elif ratio > 1.5:
                thr += 0.06

        pr = self._precision
        if pr is not None:
            if pr < 0.30:
                thr += 0.08
            elif pr < 0.50:
                thr += 0.04
            elif pr > 0.85:
                thr -= 0.06
            elif pr > 0.70:
                thr -= 0.03

        return float(np.clip(thr, self.MIN_THRESHOLD, self.MAX_THRESHOLD))

    @property
    def diagnostics(self) -> dict:
        fr = self._flag_rate
        pr = self._precision
        return {
            "threshold":       round(self.threshold, 4),
            "flag_rate":       round(fr, 4) if fr is not None else None,
            "alert_precision": round(pr, 4) if pr is not None else None,
            "stream_n":        len(self._stream),
            "reviews_n":       len(self._reviews),
        }


# Smoke test
eng = AdaptiveThresholdEngine(baseline_flag_rate=0.035)
print("Cold start (no data)       :", eng.threshold)

for _ in range(900): eng.record_transaction(False)
for _ in range(100): eng.record_transaction(True)   # 10% flag rate → 3× baseline
print("After flag-rate spike      :", eng.threshold, "← raised")

for _ in range(30): eng.record_review(True)          # 100% precision
print("After high analyst precision:", eng.threshold, "← pulled back slightly")
print("Diagnostics:", eng.diagnostics)

Cold start (no data)       : 0.5
After flag-rate spike      : 0.6 ← raised
After high analyst precision: 0.54 ← pulled back slightly
Diagnostics: {'threshold': 0.54, 'flag_rate': 0.1, 'alert_precision': 1.0, 'stream_n': 1000, 'reviews_n': 30}


## 15. Business Rules Layer
Hard rules that fire **regardless of ML score**.  
Rules are evaluated independently and each appends a structured reason string - useful for audit trails and SHAP alignment.

In [16]:
class BusinessRulesLayer:
    """
    Hard rules that fire regardless of ML score.
    Returns (flagged: bool, reasons: list[str]).
    """

    HIGH_RISK_EMAILS = {
        'protonmail.com', 'mail.com', 'guerrillamail.com',
        'tempmail.com', 'throwam.com'
    }

    def evaluate(
        self,
        score: float,
        amount: float,
        hour: int,
        email_domain: str = '',
        adaptive_threshold: float = 0.50
    ):
        reasons = []
        flagged = False

        if amount > 50_000 and score > 0.30:
            reasons.append(f'HIGH_AMOUNT: {amount:,.0f}  score={score:.3f}')
            flagged = True

        if 2 <= hour <= 5 and score > 0.25:
            reasons.append(f'NIGHT_TXN: hour={hour}  score={score:.3f}')
            flagged = True

        if email_domain.lower() in self.HIGH_RISK_EMAILS and score > 0.20:
            reasons.append(f'RISKY_EMAIL: {email_domain}  score={score:.3f}')
            flagged = True

        if score >= adaptive_threshold:
            reasons.append(f'ML_SCORE: {score:.3f} >= threshold {adaptive_threshold:.3f}')
            flagged = True

        return flagged, reasons


rules = BusinessRulesLayer()
print("Large amount  :", rules.evaluate(0.32, 75_000, 14, 'gmail.com'))
print("Night txn     :", rules.evaluate(0.28, 5_000,  3,  'gmail.com'))
print("Risky email   :", rules.evaluate(0.22, 5_000,  14, 'protonmail.com'))
print("Clean         :", rules.evaluate(0.15, 500,    14, 'gmail.com'))

Large amount  : (True, ['HIGH_AMOUNT: 75,000  score=0.320'])
Night txn     : (True, ['NIGHT_TXN: hour=3  score=0.280'])
Risky email   : (True, ['RISKY_EMAIL: protonmail.com  score=0.220'])
Clean         : (False, [])


## 16. ScoredTransaction Dataclass
Structured output for every scored transaction.  
Carries model scores, ensemble score, threshold, flag decision, reasons, and latency - all fields needed by the FastAPI backend and analyst review UI.

In [17]:
@dataclass
class ScoredTransaction:
    transaction_id: str
    lgb_score:      float
    xgb_score:      float
    iso_score:      float
    ensemble_score: float
    threshold:      float
    is_flagged:     bool
    reasons:        list
    latency_ms:     float = 0.0
    timestamp:      float = field(default_factory=time.time)

    def summary(self) -> str:
        flag = '🚨 FRAUD' if self.is_flagged else '✅ LEGIT'
        return (
            f'{flag}  |  txn={self.transaction_id}'
            f'  |  score={self.ensemble_score:.3f}'
            f'  |  threshold={self.threshold:.3f}'
            f'  |  latency={self.latency_ms:.1f}ms'
            f'  |  reasons={self.reasons}'
        )


# Demo
demo_txn = ScoredTransaction(
    transaction_id='TXN_DEMO_001',
    lgb_score=0.72, xgb_score=0.68, iso_score=0.55,
    ensemble_score=0.69, threshold=0.50,
    is_flagged=True,
    reasons=['ML_SCORE: 0.69 >= threshold 0.50', 'NIGHT_TXN: hour=3'],
    latency_ms=4.2
)
print(demo_txn.summary())

🚨 FRAUD  |  txn=TXN_DEMO_001  |  score=0.690  |  threshold=0.500  |  latency=4.2ms  |  reasons=['ML_SCORE: 0.69 >= threshold 0.50', 'NIGHT_TXN: hour=3']


## 17. Inference Pipeline
Single-row scoring function used by the async streaming engine and FastAPI backend.  
Builds a DataFrame, aligns category levels, runs all three models, applies business rules, and returns a `ScoredTransaction`.

In [18]:
# Load saved artefacts
_lgb   = joblib.load('lgb_F.pkl')
_xgb   = joblib.load('xgb_F.pkl')
_feats = joblib.load('features_F.pkl')

with open('cat_cols.json') as f:
    _cat_cols = json.load(f)

_engine = AdaptiveThresholdEngine(baseline_flag_rate=0.035)
_rules  = BusinessRulesLayer()


def score_transaction(row: dict) -> ScoredTransaction:
    t0 = time.perf_counter()

    df = pd.DataFrame([row])

    for col, cats in _cat_cols.items():
        if col in df.columns:
            df[col] = pd.Categorical(df[col], categories=cats)

    lgb_s = float(_lgb.predict_proba(df[_feats])[:, 1][0])
    xgb_s = float(_xgb.predict_proba(df[_feats])[:, 1][0])
    ens_s = W_LGB * lgb_s + W_XGB * xgb_s

    threshold = _engine.threshold
    amount    = row.get('TransactionAmt', 0)
    hour      = int((row.get('TransactionDT', 0) // 3600) % 24)
    email     = str(row.get('P_emaildomain', ''))

    flagged, reasons = _rules.evaluate(
        score=ens_s, amount=amount, hour=hour,
        email_domain=email, adaptive_threshold=threshold
    )
    latency = (time.perf_counter() - t0) * 1000

    return ScoredTransaction(
        transaction_id=str(row.get('TransactionID', '?')),
        lgb_score=lgb_s, xgb_score=xgb_s, iso_score=0.0,
        ensemble_score=ens_s, threshold=threshold,
        is_flagged=flagged, reasons=reasons, latency_ms=latency
    )


result = score_transaction(X_val.iloc[0].to_dict())
print(result.summary())

✅ LEGIT  |  txn=3459432  |  score=0.077  |  threshold=0.500  |  latency=19.5ms  |  reasons=[]


## 18. Async Streaming Engine
AsyncIO producer-consumer pattern.  
**Producer** replays historical transactions at a configurable speed.  
**Consumer** batches rows (50 at a time) for vectorised model inference - this is what enables 1000+ TPS.

In [19]:
import asyncio
class StreamMonitor:
    def __init__(self, window: int = 500):
        self._lats     = deque(maxlen=window)
        self._flags    = deque(maxlen=window)
        self.total     = 0
        self.n_flagged = 0
        self._t0       = time.perf_counter()

    def record(self, r: ScoredTransaction):
        self._lats.append(r.latency_ms)
        self._flags.append(int(r.is_flagged))
        self.total     += 1
        self.n_flagged += int(r.is_flagged)

    @property
    def tps(self):
        return self.total / max(time.perf_counter() - self._t0, 1e-6)

    @property
    def live_fraud_rate(self):
        return sum(self._flags) / max(len(self._flags), 1)

    @property
    def p95_latency(self):
        return float(np.percentile(list(self._lats), 95)) if self._lats else 0.0

    def summary(self) -> str:
        return (f'TPS={self.tps:,.0f}  |  '
                f'fraud_rate={self.live_fraud_rate:.3f}  |  '
                f'p95={self.p95_latency:.1f}ms  |  '
                f'total={self.total:,}')


async def producer(queue, rows, speed: int = 500):
    delay = 1.0 / speed
    for row in rows:
        await queue.put(row)
        await asyncio.sleep(delay)
    await queue.put(None)   # sentinel


async def consumer(queue, monitor: StreamMonitor):
    flagged_log = []
    batch       = []

    while True:
        row = await queue.get()

        if row is None:
            # Flush remaining batch
            if batch:
                df = pd.DataFrame(batch)
                for col, cats in _cat_cols.items():
                    if col in df.columns:
                        df[col] = pd.Categorical(df[col], categories=cats)
                lgb_all = _lgb.predict_proba(df[_feats])[:, 1]
                xgb_all = _xgb.predict_proba(df[_feats])[:, 1]
                ens_all = W_LGB * lgb_all + W_XGB * xgb_all
                for i, r in enumerate(batch):
                    ens_s = float(ens_all[i])
                    thr   = _engine.threshold
                    flagged, reasons = _rules.evaluate(
                        ens_s, r.get('TransactionAmt', 0),
                        int((r.get('TransactionDT', 0) // 3600) % 24),
                        str(r.get('P_emaildomain', '')), thr
                    )
                    result = ScoredTransaction(
                        transaction_id=str(r.get('TransactionID', '?')),
                        lgb_score=float(lgb_all[i]), xgb_score=float(xgb_all[i]),
                        iso_score=0.0, ensemble_score=ens_s,
                        threshold=thr, is_flagged=flagged, reasons=reasons
                    )
                    monitor.record(result)
                    _engine.record_transaction(flagged)
                    if flagged:
                        flagged_log.append(result)
            break

        batch.append(row)

        if len(batch) >= 50:
            df = pd.DataFrame(batch)
            for col, cats in _cat_cols.items():
                if col in df.columns:
                    df[col] = pd.Categorical(df[col], categories=cats)
            lgb_all = _lgb.predict_proba(df[_feats])[:, 1]
            xgb_all = _xgb.predict_proba(df[_feats])[:, 1]
            ens_all = W_LGB * lgb_all + W_XGB * xgb_all 
            for i, r in enumerate(batch):
                ens_s = float(ens_all[i])
                thr   = _engine.threshold
                flagged, reasons = _rules.evaluate(
                    ens_s, r.get('TransactionAmt', 0),
                    int((r.get('TransactionDT', 0) // 3600) % 24),
                    str(r.get('P_emaildomain', '')), thr
                )
                result = ScoredTransaction(
                    transaction_id=str(r.get('TransactionID', '?')),
                    lgb_score=float(lgb_all[i]), xgb_score=float(xgb_all[i]),
                    iso_score=0.0, ensemble_score=ens_s,
                    threshold=thr, is_flagged=flagged, reasons=reasons
                )
                monitor.record(result)
                _engine.record_transaction(flagged)
                if flagged:
                    flagged_log.append(result)
            batch = []

        queue.task_done()

    return flagged_log


async def run_stream(rows, speed: int = 500):
    queue   = asyncio.Queue(maxsize=200)
    monitor = StreamMonitor()
    prod    = asyncio.create_task(producer(queue, rows, speed))
    cons    = asyncio.create_task(consumer(queue, monitor))
    await prod
    flagged = await cons
    return monitor, flagged


# Run streaming benchmark
stream_rows              = X_val.head(2000).to_dict(orient='records')
monitor, flagged_txns    = await run_stream(stream_rows, speed=500)

print('=' * 55)
print('STREAMING BENCHMARK (2,000 transactions @ 500 TPS)')
print('=' * 55)
print(monitor.summary())
print()
print('Sample flagged transactions:')
for t in flagged_txns[:3]:
    print(' ', t.summary())

STREAMING BENCHMARK (2,000 transactions @ 500 TPS)
TPS=316  |  fraud_rate=0.016  |  p95=0.0ms  |  total=2,000

Sample flagged transactions:
  🚨 FRAUD  |  txn=3459499  |  score=0.463  |  threshold=0.500  |  latency=0.0ms  |  reasons=['NIGHT_TXN: hour=3  score=0.463']
  🚨 FRAUD  |  txn=3459504  |  score=0.464  |  threshold=0.500  |  latency=0.0ms  |  reasons=['NIGHT_TXN: hour=4  score=0.464']
  🚨 FRAUD  |  txn=3459510  |  score=0.887  |  threshold=0.500  |  latency=0.0ms  |  reasons=['NIGHT_TXN: hour=4  score=0.887', 'ML_SCORE: 0.887 >= threshold 0.500']


## 19. SHAP Explainability

SHAP (SHapley Additive exPlanations) provides per-feature contribution scores for every flagged transaction.  
- Uses `TreeExplainer` - exact SHAP values for tree models, sub-millisecond per row  
- Explains **why** the model flagged a transaction, not just the score  
- Required in regulated fintech environments (RBI guidelines require explainable fraud decisions)

In [24]:
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Build explainer on LightGBM ───────────────────────────────────────────────
explainer = shap.TreeExplainer(_lgb)
sample_df = X_val[_feats].head(200)
shap_vals = explainer.shap_values(sample_df)
sv        = shap_vals[1] if isinstance(shap_vals, list) else shap_vals

print('SHAP values computed.')
print(f'Shape: {sv.shape}  ({sv.shape[0]} transactions × {sv.shape[1]} features)')


# ── Per-transaction explanation function ──────────────────────────────────────
def explain_transaction(row_df: pd.DataFrame, top_n: int = 3) -> list:
    """
    Returns top N SHAP contributors for a single transaction row.
    row_df must already have categories aligned to training levels.
    """
    vals  = explainer.shap_values(row_df)
    sv_   = vals[1][0] if isinstance(vals, list) else vals[0]
    pairs = sorted(zip(_feats, sv_), key=lambda x: abs(x[1]), reverse=True)
    return [
        {
            'feature':   feat,
            'value':     float(row_df[feat].iloc[0]),
            'shap':      round(float(s), 4),
            'direction': '↑ fraud' if s > 0 else '↓ fraud'
        }
        for feat, s in pairs[:top_n]
    ]


# ── Select highest-scored transaction ─────────────────────────────────────────
ensemble_preds = W_LGB * lgb_preds + W_XGB * xgb_preds
high_score_idx = ensemble_preds.argmax()
row_df         = X_val[_feats].iloc[[high_score_idx]].reset_index(drop=True)
explanation    = explain_transaction(row_df)

# ── Text output (unchanged) ───────────────────────────────────────────────────
print(f'\nTransaction index : {high_score_idx}')
print(f'Ensemble score    : {ensemble_preds[high_score_idx]:.4f}')
print(f'True label        : {X_val["isFraud"].iloc[high_score_idx]}')
print('\nTop SHAP contributors:')
for e in explanation:
    bar = '█' * int(abs(e['shap']) * 30)
    print(f"  {e['feature']:25s}  val={e['value']:.3f}  shap={e['shap']:+.4f}  {e['direction']}  {bar}")


# ── 1. SHAP Summary Plot — global feature importance ─────────────────────────
fig_summary, ax_summary = plt.subplots(figsize=(10, 8))
shap.summary_plot(
    sv,
    sample_df,
    feature_names=_feats,
    plot_type='dot',        # beeswarm: shows distribution + direction
    max_display=20,
    show=False,
    plot_size=None          # let figsize control size
)
plt.title('SHAP Feature Importance — Top 20 Features\n(LightGBM · 200 validation transactions)',
          fontsize=13, fontweight='bold', pad=14)
plt.tight_layout()
plt.savefig('shap_summary.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved → shap_summary.png')


# ── 2. SHAP Waterfall Plot — single transaction breakdown ─────────────────────
# Build a shap.Explanation object required by the waterfall API
single_vals  = explainer.shap_values(row_df)
single_sv    = single_vals[1][0] if isinstance(single_vals, list) else single_vals[0]
base_value   = (explainer.expected_value[1]
                if isinstance(explainer.expected_value, (list, np.ndarray))
                else explainer.expected_value)

explanation_obj = shap.Explanation(
    values        = single_sv,
    base_values   = base_value,
    data          = row_df.iloc[0].values,
    feature_names = _feats
)

fig_wf, ax_wf = plt.subplots(figsize=(10, 8))
shap.plots.waterfall(explanation_obj, max_display=15, show=False)
plt.title(
    f'SHAP Waterfall — Transaction #{high_score_idx}\n'
    f'Ensemble score: {ensemble_preds[high_score_idx]:.4f}  |  '
    f'True label: {int(X_val["isFraud"].iloc[high_score_idx])}',
    fontsize=12, fontweight='bold', pad=14
)
plt.tight_layout()
plt.savefig('shap_waterfall.png', dpi=180, bbox_inches='tight')
plt.show()
print('Saved → shap_waterfall.png')

SHAP values computed.
Shape: (200, 98)  (200 transactions × 98 features)

Transaction index : 85892
Ensemble score    : 0.9996
True label        : 1

Top SHAP contributors:
  V149                       val=4.000  shap=+1.4602  ↑ fraud  ███████████████████████████████████████████
  V258                       val=4.000  shap=+1.3257  ↑ fraud  ███████████████████████████████████████
  V156                       val=3.000  shap=+1.2310  ↑ fraud  ████████████████████████████████████
Saved → shap_summary.png
Saved → shap_waterfall.png


## 20. Final Benchmark Report
> **Run this cell last, after all training and streaming cells are complete.**

Formal benchmark across throughput, latency percentiles, and model quality metrics.  
These are the numbers that go on the resume.

In [23]:
from sklearn.metrics import f1_score, precision_score, recall_score
import numpy as np

print('=' * 60)
print('FRAUD DETECTION ENGINE — FINAL BENCHMARK REPORT')
print('=' * 60)

# ── Model Quality ────────────────────────────────────────────────────────
ensemble_preds = W_LGB * lgb_preds + W_XGB * xgb_preds

# Find optimal F1 threshold
best_f1, best_thresh = 0, 0.5
for t in np.arange(0.20, 0.80, 0.01):
    binary = (ensemble_preds >= t).astype(int)
    f = f1_score(X_val['isFraud'], binary, zero_division=0)
    if f > best_f1:
        best_f1, best_thresh = f, t

binary_preds = (ensemble_preds >= best_thresh).astype(int)

prauc     = average_precision_score(X_val['isFraud'], ensemble_preds)
f1        = f1_score(X_val['isFraud'], binary_preds)
precision = precision_score(X_val['isFraud'], binary_preds, zero_division=0)
recall    = recall_score(X_val['isFraud'], binary_preds)
fnr       = 1 - recall   # false negative rate: missed fraud

print(f'\n── Model Quality (Validation Set) ──────────────────────')
print(f'  Ensemble PR-AUC              : {prauc:.4f}')
print(f'  F1 Score @ optimal threshold : {f1:.4f}  (threshold={best_thresh:.2f})')
print(f'  Precision                    : {precision:.4f}')
print(f'  Recall                       : {recall:.4f}')
print(f'  False Negative Rate          : {fnr:.4f}  ← missed fraud')
print(f'  Val set size                 : {len(X_val):,}')
print(f'  Fraud cases in val           : {X_val["isFraud"].sum():,}')

# ── Throughput Benchmark ─────────────────────────────────────────────────
print(f'\n── Throughput Benchmark (batch vectorised) ─────────────')
for n in [1_000, 5_000, 10_000, 50_000]:
    sample = X_val.head(n)
    df = sample.copy()
    for col, cats in _cat_cols.items():
        if col in df.columns:
            df[col] = pd.Categorical(df[col], categories=cats)
    t0      = time.perf_counter()
    l_out   = _lgb.predict_proba(df[_feats])[:, 1]
    x_out   = _xgb.predict_proba(df[_feats])[:, 1]
    elapsed = time.perf_counter() - t0
    tps     = n / elapsed
    print(f'  {n:>6,} rows  →  {tps:>8,.0f} TPS  ({elapsed*1000:.1f} ms total)')

# ── Single-Row Latency ───────────────────────────────────────────────────
print(f'\n── Single-Row Inference Latency (100 runs) ─────────────')
lats = []
for i in range(100):
    row = X_val.iloc[i % len(X_val)].to_dict()
    r   = score_transaction(row)
    lats.append(r.latency_ms)

print(f'  P50 : {np.percentile(lats, 50):.2f} ms')
print(f'  P95 : {np.percentile(lats, 95):.2f} ms')
print(f'  P99 : {np.percentile(lats, 99):.2f} ms')
print(f'  Mean: {np.mean(lats):.2f} ms')

# ── Ensemble Weights ─────────────────────────────────────────────────────
print(f'\n── Ensemble Configuration ──────────────────────────────')
print(f'  LightGBM weight  : {W_LGB:.2f}')
print(f'  XGBoost weight   : {W_XGB:.2f}')
print(f'  Dataset          : IEEE-CIS ({len(train):,} transactions)')

# ── Streaming Engine (from earlier cell) ────────────────────────────────
try:
    print(f'\n── Async Streaming Engine ──────────────────────────────')
    print(f'  {monitor.summary()}')
except NameError:
    print('  (Run the streaming cell above first)')

print('\n' + '=' * 60)
print('END OF BENCHMARK')
print('=' * 60)

FRAUD DETECTION ENGINE — FINAL BENCHMARK REPORT

── Model Quality (Validation Set) ──────────────────────
  Ensemble PR-AUC              : 0.6336
  F1 Score @ optimal threshold : 0.6153  (threshold=0.20)
  Precision                    : 0.7144
  Recall                       : 0.5404
  False Negative Rate          : 0.4596  ← missed fraud
  Val set size                 : 118,108
  Fraud cases in val           : 4,064

── Throughput Benchmark (batch vectorised) ─────────────
   1,000 rows  →    12,201 TPS  (82.0 ms total)
   5,000 rows  →    14,618 TPS  (342.0 ms total)
  10,000 rows  →    15,675 TPS  (638.0 ms total)
  50,000 rows  →    17,014 TPS  (2938.8 ms total)

── Single-Row Inference Latency (100 runs) ─────────────
  P50 : 13.88 ms
  P95 : 15.09 ms
  P99 : 22.21 ms
  Mean: 14.16 ms

── Ensemble Configuration ──────────────────────────────
  LightGBM weight  : 0.45
  XGBoost weight   : 0.55
  Dataset          : IEEE-CIS (590,540 transactions)

── Async Streaming Engine ──────────